In [151]:
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal

In [176]:
insurance_loader = TextLoader("./insurance.txt", encoding = "utf-8")  
insurance_documents = insurance_loader.load()

insurance_documents[0].metadata = {
    "source": "insurance.txt",
    "domain": "Insurance",
    "category": "P&C",
    "document_type": "training"
}
# print(insurance_documents)

In [177]:
psycology_loader = TextLoader("./psychology.txt", encoding = "utf-8")  
psycology_documents = psycology_loader.load()

psycology_documents[0].metadata = {
    "source": "psychology.txt",
    "domain": "Psychology",
    "category": "Mental Health",
    "document_type": "training"
}
# print(psycology_documents)

In [178]:
artifical_intelligence_loader = TextLoader("./artifical_intelligence.txt", encoding = "utf-8")  
artifical_intelligence_documents = artifical_intelligence_loader.load()

artifical_intelligence_documents[0].metadata = {
    "source": "artifical_intelligence.txt",
    "domain": "AI",
    "category": "AI & ML",
    "document_type": "training"
}
# print(artifical_intelligence_documents)

In [ ]:
class DataPipeline():
    def __init__(self,documents):
        self.documents = documents

    def chunk_emb(self,documents):
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100
        )
        chunks = splitter.split_documents(
            self.documents
        )
        embeddings = OllamaEmbeddings(
            model="mxbai-embed-large:latest"
        )
        vector_db = FAISS.from_documents(
            documents=chunks,
            embedding=embeddings,
            distance_strategy=DistanceStrategy.COSINE
        )

        return vector_db

In [156]:
# insurance = DataPipeline(insurance_documents)
# insurance_emb = insurance.chunk_emb(insurance_documents)
# # print(insurance_emb)
# print(len(insurance_emb.index_to_docstore_id))


In [157]:
# psycology = DataPipeline(psycology_documents)
# psycology_emb = psycology.chunk_emb(psycology_documents)
# # print(psycology_emb)
# print(len(psycology_emb.index_to_docstore_id))

In [158]:
# ai = DataPipeline(artifical_intelligence_documents)
# ai_emb = ai.chunk_emb(artifical_intelligence_documents)
# # print(ai_emb)
# print(len(ai_emb.index_to_docstore_id))

In [159]:
# insurance_emb.merge_from(psycology_emb)

# insurance_emb.merge_from(ai_emb)

In [160]:

# final_emb = insurance_emb
# final_emb.save_local(
#     "vector_embedding_db"
# )

In [161]:
embeddings = OllamaEmbeddings(
            model="mxbai-embed-large:latest")
vec_db = FAISS.load_local("vector_embedding_db",
                                  embeddings=embeddings,
                                  allow_dangerous_deserialization = True
)

In [162]:
def retrieve_documents(inputs):

    topic = inputs["classify"]
    query = inputs["user_query"]
    # print(topic)

    retriever = vec_db.as_retriever(
        search_kwargs={
            "k":5,
            "filter":{
                "domain": topic
            }
        }
    )

    docs = retriever.invoke(query)

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [163]:
class Classify(BaseModel):
    user_query : str  = Field(description="The exact user query received")
    classify : Literal["Insurance", "AI", "Psychology"] = Field(description="Classification of the user query.")

In [164]:
new_parser = PydanticOutputParser(pydantic_object=Classify)

In [165]:
format_instructions = new_parser.get_format_instructions()

In [166]:
classify_prompt = ChatPromptTemplate.from_template("""
                                          You are a classifier expert. Based on user query classify the query into one of the below
                                          ['AI','Insurance','Psychology']. 
                                        Do Not repeat this schema or any descriptions. Only output the final JSON
                                                   
                                                   Output format:
                                          {format_instructions}
                                          
                                          User query = {user_query}
                                          """,
    partial_variables={
        "format_instructions": new_parser.get_format_instructions()
    })

In [167]:
prompt = ChatPromptTemplate.from_template("""
                                          You are a expert summarizer. Baesd on user question and available context,
                                           provide a detailed pointiwse explanation. Be short and consince in your response in 4-5 lines and give response in plain text.
                                          
                                          User query = {user_query}

                                          Available Context = {relevant_document}
                                          """)


In [168]:
llm = OllamaLLM(model='llama3.2:3b', temperature=0.7, format='json')

In [169]:
classifier = (
    classify_prompt
    | llm
    | new_parser
)

pipeline = (
    RunnableLambda(
        lambda x: {
            "user_query": x["user_query"],
            "classify": classifier.invoke({
                "user_query": x["user_query"],
                "format_instructions":  x["format_instructions"]
            }).classify
        }
    )

    |

    RunnableLambda(
        lambda x: {
            "user_query": x["user_query"],
            "relevant_document": retrieve_documents(x)
        }
    )

    |

    prompt
    
    |

    llm
    
    |

    StrOutputParser()
)


In [170]:
message = pipeline.invoke({"user_query":"What is P&C Insurance?",
"format_instructions":format_instructions
})
print(message)

{ "P&C Insurance":{ 
  "Description":"A package policy providing multiple types of general insurance coverage under a single product for various business risks", 
  "Types of Coverage":"- Business Interruption Insurance",
  "- Burglary Insurance (protects against attempted robberies and theft of valuables within the property) ":[
    "Business Property Insurance",
    "Marine and Cargo Insurance",
    "Liability Insurance (protects against lawsuits)",
    "Workers Compensation Insurance"
  ],
  "Benefits":{ 
    "Financial Coverage Against Losses and Damage to Property":[
      "Recover from Business Interruption"
      ],
    "Protection of Valuables Within the Property":[
      "Burglary Insurance",
      "Theft of Valuables"
     ],
    "Mitigates Various Business Risks":[
      "Property Damage",
      "Liability for Accidents or Injuries to Employees or Customers",
      "Loss of Business Income due to Interruption"
     ]
  }
} }


In [175]:
message = pipeline.invoke({"user_query":"Tell me brief about Beginning of experimental psychology",
"format_instructions":format_instructions
})
print(message)

{
  "Summary": "Experimental psychology began with Wilhelm Wundt's establishment of the first psychological laboratory in Leipzig University (c. 1880), focusing on breaking down mental processes into basic components. This marked a shift from non-experimental approaches like phrenology and psychoanalysis.",
  "Key Dates": "1873: Ivan Sechenov's essay 'Who Is to Develop Psychology and How?' introduced the idea of brain reflexes and deterministic behavior.",
  "Influential Figures": "Wilhelm Wundt, James McKeen Cattell, Hermann von Helmholtz, Horacio G. Piñero, and Ivan Sechenov contributed to the development of experimental psychology."
}
